# Primary-Signal Portfolio Performance — vol-targeted vs raw, full-sample & OOS

How does the **primary trading signal** perform as a *standalone* long/short portfolio, before any
meta-model filtering? This is the baseline the meta-model must eventually justify itself against.

We take the released signal `s ∈ {-1,0,+1}` (daily, 11 futures, 2020-01-03 → 2022-06-30) and the
asset price series, and build a daily portfolio under a **2×2** of construction choices:

| # | book | weighting | portfolio-level overlay |
|---|------|-----------|--------------------------|
| 1 | `raw`            | `wᵢ = sᵢ` (equal-weight sign)                 | none |
| 2 | `inv_vol_10pct`  | `wᵢ = sᵢ·σ_tgt/σᵢ` (each asset → 10% ann vol) | none |
| 3 | `raw_port10pct`  | `wᵢ = sᵢ`                                     | scale book → 10% ann vol |
| 4 | `inv_vol_port10` | `wᵢ = sᵢ·σ_tgt/σᵢ`                            | scale book → 10% ann vol |

**Conventions (load-bearing):**
- **No lookahead (enter at t+1, CLAUDE.md §7).** The position held *during* day `t` uses the signal
  observed at `t-1` (`sig.shift(1)`) and an **ex-ante** vol forecast for `t` (info ≤ `t-1`).
- **Simple returns**, close-to-close, on each instrument's own dense calendar; structural holiday NaNs
  are kept (a closed venue ⇒ that sleeve is flat / in cash that day, never forward-filled).
- Equity curve `= ∏(1+rₚ)`; Sharpe `= mean/std·√252`; ann vol `= std·√252`; MDD on the equity curve;
  turnover `= mean_t Σᵢ|Wᵢ,t − Wᵢ,t-1|` (effective capital weights, incl. any leverage overlay).
- **Vol estimator is configurable** (`VOL_ESTIMATOR`): GARCH(1,1) (default), EWMA, or rolling std.

**Outputs:** PnL-path plots (4 books × 3 periods + per-instrument legs) and a metrics table written to
`results/primary_signal_portfolio_metrics.csv`.

## §0 — Setup & configuration

In [ ]:
%matplotlib inline
import warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
np.random.seed(42)

from stml.io import _find_repo_root, load_clean_data, load_returns_panel
from stml.na_checks import native_returns

ROOT = _find_repo_root(Path.cwd().resolve()); RESULTS = ROOT / "results"

# ---- configuration -------------------------------------------------------------
VOL_ESTIMATOR = "garch"        # "garch" (default) | "ewma" | "rolling"
SIGMA_TGT_ANN = 0.10           # 10% annualized volatility target
ANN = np.sqrt(252.0)           # daily -> annual scaling
FIT_END = pd.Timestamp("2021-07-01")   # GARCH params fit on returns <= this (FE-train boundary, §6)
ROLL_WIN, EWMA_SPAN = 20, 60   # windows for the rolling / EWMA estimators
LEV_CAP = 5.0                  # cap on the portfolio-level leverage overlay
print("vol estimator:", VOL_ESTIMATOR, "| target ann vol:", SIGMA_TGT_ANN)

## §1 — Load data & align the grid

`signals` is wide `date × 11` in `{-1,0,+1}`. We build two return views:
- **cross-sectional wide panel** (`ret_w`) for portfolio PnL — structural NaNs mark non-trading sleeves;
- **long dense per-instrument** (`ret_long`) for vol estimation — each instrument on its own calendar.

The signal is lagged one day (`sig_lag`) so day-`t` positions use only information through `t-1`.

In [ ]:
ohlcv, signals = load_clean_data()
INSTRUMENTS = [c for c in signals.columns if c != "date"]    # canonical 11, signal-file order

sig_w = signals.set_index("date")[INSTRUMENTS].sort_index()
WIN_START, WIN_END = sig_w.index.min(), sig_w.index.max()
print("signal window:", WIN_START.date(), "->", WIN_END.date(), "| trading dates:", len(sig_w))

# simple returns: wide cross-sectional (PnL) + long dense per-instrument (vol)
ret_wide_full = load_returns_panel(kind="simple")[INSTRUMENTS]
ret_long = native_returns(ohlcv, kind="simple")[["date", "instrument", "ret"]]

ret_w = ret_wide_full.loc[
    (ret_wide_full.index >= WIN_START) & (ret_wide_full.index <= WIN_END), INSTRUMENTS
].reindex(sig_w.index)

sig_lag = sig_w.shift(1)        # enter at t+1: position for day t decided at t-1
print("non-zero signal-days per instrument (lagged):")
print((sig_lag.abs() > 0).sum().to_string())

## §2 — Ex-ante volatility estimation

The vol forecast for day `t` must use only information through `t-1`. GARCH(1,1) is fit **once on
returns ≤ `FIT_END`** (the FE-train boundary, so the 2022 OOS windows are genuinely out-of-sample for
the vol model too), then the conditional-variance recursion `hₜ = ω + α·ε²ₜ₋₁ + β·hₜ₋₁` is filtered
forward — `hₜ` depends only on returns through `t-1`, so `σₜ` is a true one-step-ahead forecast. The
EWMA / rolling estimators are shifted one day for the same reason.

In [ ]:
def _garch_sigma_ann(ret_dense: pd.Series, fit_end: pd.Timestamp) -> pd.Series:
    """Ex-ante 1-step GARCH(1,1) conditional vol (annualized), params fit on returns <= fit_end."""
    from arch import arch_model
    r = ret_dense.dropna().sort_index()
    rp = r * 100.0                                   # percent scale -> solver stability
    train = rp[rp.index <= fit_end]
    res = arch_model(train, mean="Zero", vol="GARCH", p=1, q=1, dist="normal").fit(disp="off")
    w, a, b = res.params["omega"], res.params["alpha[1]"], res.params["beta[1]"]
    eps2 = rp.to_numpy() ** 2
    h = np.empty(len(rp)); h[0] = float(np.nanvar(train.to_numpy()))   # robust init
    for t in range(1, len(rp)):
        h[t] = w + a * eps2[t - 1] + b * h[t - 1]    # h_t uses eps_{t-1} -> forecast for day t
    return pd.Series(np.sqrt(h) / 100.0 * ANN, index=rp.index)

def _series_vol(x: pd.Series, estimator: str) -> pd.Series:
    """Ex-ante annualized vol of one return series (same family as VOL_ESTIMATOR)."""
    x = x.dropna()
    if estimator == "garch":
        try:
            return _garch_sigma_ann(x, FIT_END)
        except Exception as exc:                      # non-convergence fallback
            print("  GARCH fallback -> ewma:", exc)
            return (x.ewm(span=EWMA_SPAN).std() * ANN).shift(1)
    if estimator == "ewma":
        return (x.ewm(span=EWMA_SPAN).std() * ANN).shift(1)
    if estimator == "rolling":
        return (x.rolling(ROLL_WIN, min_periods=ROLL_WIN).std() * ANN).shift(1)
    raise ValueError(f"unknown estimator {estimator!r}")

def vol_forecast(ret_long: pd.DataFrame, mode: str, instruments) -> pd.DataFrame:
    """Ex-ante annualized vol panel (date × instrument)."""
    return pd.DataFrame({
        inst: _series_vol(
            ret_long.loc[ret_long.instrument == inst].set_index("date")["ret"].sort_index(), mode
        )
        for inst in instruments
    }).sort_index()

vol_full = vol_forecast(ret_long, VOL_ESTIMATOR, INSTRUMENTS)
vol_w = vol_full.reindex(sig_w.index)[INSTRUMENTS]
print("per-instrument mean ex-ante annualized vol over the window:")
print(vol_w.mean().round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
for inst in ["es1s", "cl1s", "gc1s"]:
    ax.plot(vol_w.index, vol_w[inst], label=inst, lw=1.1)
ax.axhline(SIGMA_TGT_ANN, color="k", ls="--", lw=0.8, label="10% target")
ax.set_title(f"Ex-ante annualized volatility forecast ({VOL_ESTIMATOR})")
ax.set_ylabel("ann. vol"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## §3 — Build the four books

`N` = number of instruments with a valid price that day (flat sleeves hold cash, so the book is an
equal-weight portfolio of 11 sleeves). The inverse-vol weighting scales each sleeve to 10% ann vol;
the portfolio-level overlay then scales the *whole book* by `Lₜ = σ_tgt/σ_port,t` (ex-ante, capped).

In [ ]:
def build_book(sig_lag, ret_w, vol_w, weighting, overlay, *, estimator):
    """Return (portfolio_return: Series, weights: DataFrame of effective capital weights)."""
    valid = ret_w.notna()
    n_valid = valid.sum(axis=1).replace(0, np.nan)
    if weighting == "raw":
        gross = sig_lag.copy()
    elif weighting == "inverse_vol":
        gross = sig_lag * (SIGMA_TGT_ANN / vol_w)
    else:
        raise ValueError(weighting)
    gross = gross.where(valid)                       # no weight where no price
    W = gross.div(n_valid, axis=0)                   # equal-weight across sleeves
    base_ret = (W * ret_w).sum(axis=1, min_count=1)  # portfolio simple return
    if not overlay:
        return base_ret, W
    sp = _series_vol(base_ret, estimator)            # ex-ante vol of the book's own return series
    L = (SIGMA_TGT_ANN / sp).clip(upper=LEV_CAP)
    L = L.reindex(base_ret.index).fillna(1.0)        # unlevered during warm-up
    return base_ret * L, W.mul(L, axis=0)

BOOKS = {
    "raw":             ("raw",         False),   # 1: equal-weight signs
    "inv_vol_10pct":   ("inverse_vol", False),   # 2: asset-by-asset 10% vol target
    "raw_port10pct":   ("raw",         True),    # 3: portfolio-level 10% overlay
    "inv_vol_port10":  ("inverse_vol", True),    # 4: both
}
book_ret, book_w = {}, {}
for name, (wt, ov) in BOOKS.items():
    book_ret[name], book_w[name] = build_book(sig_lag, ret_w, vol_w, wt, ov, estimator=VOL_ESTIMATOR)
book_ret = pd.DataFrame(book_ret)
print(book_ret.describe().T[["count", "mean", "std", "min", "max"]].round(5))

In [ ]:
# sanity: portfolio-level overlay should pull realized vol toward 10%
realized = book_ret.std() * ANN
print("full-period realized annualized vol:")
print(realized.round(3).to_string())
assert abs(realized["raw_port10pct"] - 0.10)  <= abs(realized["raw"] - 0.10) + 1e-9
assert abs(realized["inv_vol_port10"] - 0.10) <= abs(realized["inv_vol_10pct"] - 0.10) + 1e-9
print("OK: portfolio-level overlay moves realized vol toward the 10% target")

## §4 — Performance metrics

`perf_stats` operates on a daily return series + its weight frame (the project's `nav_sharpe` is
event/overlap-based, not a daily time series, so we compute fresh here while keeping its `√252`
convention). Sharpe is invariant to a *constant* leverage rescale — asserted below — which is exactly
why the portfolio-level overlay changes Sharpe only through its *time-varying* leverage.

In [ ]:
PERIODS = {
    "full":                (WIN_START,                     WIN_END),
    "oos_2022h1":          (pd.Timestamp("2022-01-01"),    WIN_END),
    "oos_from_2021-10-21": (pd.Timestamp("2021-10-21"),    WIN_END),
}

def perf_stats(ret: pd.Series, weights: pd.DataFrame, start, end) -> dict:
    r = ret.loc[start:end].dropna()
    if len(r) < 2:
        return dict(n=len(r), sharpe=np.nan, cum_return=np.nan, ann_vol=np.nan, mdd=np.nan, turnover=np.nan)
    equity = (1 + r).cumprod()
    dW = weights.reindex(r.index).diff().abs().sum(axis=1, min_count=1)
    return dict(
        n=int(len(r)),
        sharpe=float(r.mean() / r.std() * ANN),
        cum_return=float(equity.iloc[-1] - 1.0),
        ann_vol=float(r.std() * ANN),
        mdd=float((equity / equity.cummax() - 1.0).min()),
        turnover=float(dW.mean()),
    )

# leverage invariance: a constant rescale leaves Sharpe unchanged
_a = perf_stats(book_ret["raw"],        book_w["raw"],        WIN_START, WIN_END)["sharpe"]
_b = perf_stats(book_ret["raw"] * 3.0,  book_w["raw"] * 3.0,  WIN_START, WIN_END)["sharpe"]
assert abs(_a - _b) < 1e-9, (_a, _b)
print(f"OK leverage-invariance: Sharpe(raw)={_a:.3f} == Sharpe(3x raw)={_b:.3f}")

## §5 — PnL paths

Equity curves (rebased to 1 at each window start) for the four books across the full sample and the
two OOS windows, then a per-instrument panel of the inverse-vol legs (low-power names `cl1s` / `ho1s`
/ `ng1s` highlighted, per CLAUDE.md §7 — don't bury the thin instruments).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (pname, (s, e)) in zip(axes, PERIODS.items()):
    for nm in BOOKS:
        r = book_ret[nm].loc[s:e].dropna()
        eq = (1 + r).cumprod()
        ax.plot(eq.index, eq.values, label=nm, lw=1.2)
    ax.axhline(1.0, color="k", lw=0.6, ls=":")
    ax.set_title(f"{pname}\n{pd.Timestamp(s).date()} -> {pd.Timestamp(e).date()}", fontsize=10)
    ax.set_ylabel("equity (rebased = 1)"); ax.legend(fontsize=7); ax.tick_params(labelsize=8)
fig.suptitle(f"Primary-signal portfolio PnL paths — vol estimator: {VOL_ESTIMATOR}", y=1.04)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 13)); axes = axes.ravel()
low_power = {"cl1s", "ho1s", "ng1s"}
for ax, inst in zip(axes, INSTRUMENTS):
    leg = (sig_lag[inst] * (SIGMA_TGT_ANN / vol_w[inst]) * ret_w[inst]).dropna()
    eq = (1 + leg).cumprod()
    ax.plot(eq.index, eq.values, lw=1.1, color=("crimson" if inst in low_power else "steelblue"))
    ax.axhline(1.0, color="k", lw=0.6, ls=":")
    ax.set_title(inst + ("  (low-power)" if inst in low_power else ""), fontsize=9)
    ax.tick_params(labelsize=7)
for ax in axes[len(INSTRUMENTS):]:
    ax.axis("off")
fig.suptitle("Per-instrument inverse-vol leg equity (each sleeve targeted to 10% ann vol)", y=1.0)
plt.tight_layout(); plt.show()

## §6 — Metrics table

Sharpe / cumulative return / annualized vol / MDD / turnover for each book × period, written to
`results/primary_signal_portfolio_metrics.csv` (tagged with the active `vol_estimator`).

In [ ]:
rows = []
for nm in BOOKS:
    for pname, (s, e) in PERIODS.items():
        rows.append({"portfolio": nm, "period": pname, **perf_stats(book_ret[nm], book_w[nm], s, e)})
metrics = pd.DataFrame(rows)
metrics.insert(0, "vol_estimator", VOL_ESTIMATOR)
assert len(metrics) == 12, len(metrics)

out_path = RESULTS / "primary_signal_portfolio_metrics.csv"
metrics.round(6).to_csv(out_path, index=False)
print("wrote", out_path, "| rows", len(metrics))

show = metrics.copy()
for c in ["cum_return", "ann_vol", "mdd"]:
    show[c] = (show[c] * 100).round(2)
show["sharpe"] = show["sharpe"].round(2); show["turnover"] = show["turnover"].round(3)
show = show.rename(columns={"cum_return": "cum_ret_%", "ann_vol": "ann_vol_%", "mdd": "mdd_%"})
display(show.set_index(["portfolio", "period"])[["sharpe", "cum_ret_%", "ann_vol_%", "mdd_%", "turnover", "n"]])

## §7 — Interpretation

Read the table above against these structural facts (they hold by construction, independent of the
particular sample):

- **Sharpe is leverage-invariant under a *constant* rescale** (asserted in §4). So `raw` vs
  `raw_port10pct` differ in Sharpe **only** through the *time-varying* leverage `Lₜ` the portfolio-level
  overlay applies — it de-levers into high-vol regimes and levers up into calm ones. The overlay's real
  job is to put the book at a **known ~10% vol** so cumulative return / MDD are comparable across books.
- **Inverse-vol weighting genuinely reweights the cross-section** (each sleeve to 10% ann vol), so
  `inv_vol_*` Sharpe differs from `raw` Sharpe for an economic reason: it stops a few high-vol legs
  (energy: `cl1s`, `ng1s`) from dominating risk, and gives quiet legs (equity index) a fair budget.
- **Turnover** rises from `raw` (only signal flips) → inverse-vol (daily σ drift resizes every leg) →
  portfolio-overlay (adds book-level leverage churn). Higher turnover ⇒ more cost sensitivity.
- **OOS windows.** `oos_2022h1` is the clean Jan–Jun 2022 hold-out (the deliverable period);
  `oos_from_2021-10-21` adds the late-2021 tail. Compare Sharpe/MDD there to the full sample to see
  whether the signal's risk-adjusted return *persists* out of sample or was a 2020–21 artifact.

> **Caveat.** This is the *raw* primary signal sized mechanically — no meta-model filtering yet. It is
> the baseline the meta-model (the graded deliverable) must beat *blindly* (CLAUDE.md §5 Part 3). Switch
> `VOL_ESTIMATOR` to `"ewma"` / `"rolling"` and re-run to confirm the ranking is not an artifact of the
> GARCH vol model.